In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

In [ ]:
# Cargar variables de entorno desde el archivo .env
ruta_env = Path("../.env").resolve()
load_dotenv(dotenv_path=ruta_env)
# Stage
cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)


# DW

cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    f"TrustServerCertificate=yes;"
)

engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

In [10]:
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")

consulta_producto = f"""
SELECT
    producto_id,
    codigo,
    nombre,
    precio,
    costo,
    activo
FROM {esquema_stage}.Producto
"""

df_producto = pd.read_sql(
    consulta_producto,
    engine_stage
)

print("Productos extraídos:", len(df_producto))

df_producto.head()

Productos extraídos: 72


,producto_id,codigo,nombre,precio,costo,activo
0,1,FRK-0001,Figuras articulada The Legend of Zelda - Modelo 1,190.0,121.60,True
1,2,FRK-0002,Figuras articulada Pokémon - Modelo 2,420.0,260.40,True
2,3,FRK-0003,Figuras articulada El Señor de los Anillos - M...,380.0,224.20,True
3,4,FRK-0004,Funko Pop One Piece - Modelo 1,140.0,92.40,True
4,5,FRK-0005,Funko Pop Dragon Ball - Modelo 2,225.0,137.25,True


In [11]:
# Eliminar productos sin identificador.
df_producto = df_producto.dropna(
    subset=["producto_id"]
)

# Eliminar posibles duplicados por clave natural.
df_producto = df_producto.drop_duplicates(
    subset=["producto_id"]
)

# Limpiar espacios innecesarios.
df_producto["codigo"] = df_producto["codigo"].str.strip()
df_producto["nombre"] = df_producto["nombre"].str.strip()

print("Productos después de la limpieza:", len(df_producto))

Productos después de la limpieza: 72


In [12]:
df_producto = df_producto.rename(
    columns={
        "producto_id": "producto_id_origen"
    }
)

df_producto.head()

,producto_id_origen,codigo,nombre,precio,costo,activo
0,1,FRK-0001,Figuras articulada The Legend of Zelda - Modelo 1,190.0,121.60,True
1,2,FRK-0002,Figuras articulada Pokémon - Modelo 2,420.0,260.40,True
2,3,FRK-0003,Figuras articulada El Señor de los Anillos - M...,380.0,224.20,True
3,4,FRK-0004,Funko Pop One Piece - Modelo 1,140.0,92.40,True
4,5,FRK-0005,Funko Pop Dragon Ball - Modelo 2,225.0,137.25,True


In [20]:
esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")

df_productos_existentes = pd.read_sql(
    f"""
    SELECT producto_id_origen
    FROM {esquema_dw}.DimProducto
    """,
    engine_dw
)

print(
    "Productos existentes en el DW:",
    len(df_productos_existentes)
)

Productos existentes en el DW: 72


In [21]:
df_productos_nuevos = df_producto[
    ~df_producto["producto_id_origen"].isin(
        df_productos_existentes["producto_id_origen"]
    )
].copy()

print(
    "Productos nuevos:",
    len(df_productos_nuevos)
)

Productos nuevos: 0


In [22]:
if not df_productos_nuevos.empty:

    df_productos_nuevos.to_sql(
        name="DimProducto",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )

    print(
        f"{len(df_productos_nuevos)} productos nuevos cargados."
    )

else:
    print("No existen productos nuevos para cargar.")

No existen productos nuevos para cargar.


In [23]:
resultado_producto = pd.read_sql(
    f"""
    SELECT TOP 10 *
    FROM {esquema_dw}.DimProducto
    ORDER BY producto_sk
    """,
    engine_dw
)

resultado_producto

,producto_sk,producto_id_origen,codigo,nombre,precio,costo,activo,fecha_carga
0,1,1,FRK-0001,Figuras articulada The Legend of Zelda - Modelo 1,190.0,121.60,True,2026-10-09 13:02:12.132138
1,2,2,FRK-0002,Figuras articulada Pokémon - Modelo 2,420.0,260.40,True,2026-10-09 13:02:12.132138
2,3,3,FRK-0003,Figuras articulada El Señor de los Anillos - M...,380.0,224.20,True,2026-10-09 13:02:12.132138
3,4,4,FRK-0004,Funko Pop One Piece - Modelo 1,140.0,92.40,True,2026-10-09 13:02:12.132138
4,5,5,FRK-0005,Funko Pop Dragon Ball - Modelo 2,225.0,137.25,True,2026-10-09 13:02:12.132138
5,6,6,FRK-0006,Funko Pop Marvel - Modelo 3,175.0,85.75,True,2026-10-09 13:02:12.132138
6,7,7,FRK-0007,Estatua Super Mario - Modelo 1,380.0,224.20,True,2026-10-09 13:02:12.132138
7,8,8,FRK-0008,Estatua Super Mario - Modelo 2,745.0,394.85,True,2026-10-09 13:02:12.132138
8,9,9,FRK-0009,Estatua DC Comics - Modelo 3,565.0,372.90,True,2026-10-09 13:02:12.132138
9,10,10,FRK-0010,Manga Marvel - Modelo 1,55.0,30.80,True,2026-10-09 13:02:12.132138
